In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
project_folder = Path('/content/drive/MyDrive/Cyclistic Case Study')
data_folder = project_folder / '01 Original Data'

In [ ]:
csv_files = sorted(data_folder.glob('*.csv'))

print(f'Number of CSV files found: {len(csv_files)}')

for file in csv_files:
    print(file.name)

In [ ]:
dataframes = []

for file in csv_files:
    monthly_data = pd.read_csv(file)
    monthly_data["source_file"] = file.name
    dataframes.append(monthly_data)

rides = pd.concat(dataframes, ignore_index=True)

print("Files combined:", len(dataframes))
print("Total rows:", len(rides))
print("Total columns:", len(rides.columns))

In [ ]:
print(rides.columns.tolist())
print()
rides.info()

In [ ]:
rides["started_at"] = pd.to_datetime(
    rides["started_at"],
    errors="coerce"
)

rides["ended_at"] = pd.to_datetime(
    rides["ended_at"],
    errors="coerce"
)

print(rides[["started_at", "ended_at"]].dtypes)
print()
print("Invalid started_at values:", rides["started_at"].isna().sum())
print("Invalid ended_at values:", rides["ended_at"].isna().sum())

In [ ]:
duplicate_ride_ids = rides["ride_id"].duplicated().sum()

print("Duplicate ride_id values:", duplicate_ride_ids)

In [ ]:
rows_before = len(rides)

rides = rides.drop_duplicates(subset="ride_id", keep="first").copy()

rows_after = len(rides)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Duplicate rows removed:", rows_before - rows_after)

In [ ]:
missing_values = rides.isna().sum().sort_values(ascending=False)

print(missing_values)

In [ ]:
rides["ride_length_minutes"] = (
    rides["ended_at"] - rides["started_at"]
).dt.total_seconds() / 60

rides["day_of_week"] = rides["started_at"].dt.day_name()
rides["month"] = rides["started_at"].dt.month_name()
rides["hour"] = rides["started_at"].dt.hour
rides["date"] = rides["started_at"].dt.date

print(rides[
    [
        "started_at",
        "ended_at",
        "ride_length_minutes",
        "day_of_week",
        "month",
        "hour"
    ]
].head())

In [ ]:
print(rides["ride_length_minutes"].describe())

print()
print(
    "Rides with duration of 0 minutes or less:",
    (rides["ride_length_minutes"] <= 0).sum()
)

print(
    "Rides longer than 24 hours:",
    (rides["ride_length_minutes"] > 1440).sum()
)

In [ ]:
rows_before_duration_cleaning = len(rides)

rides = rides[
    (rides["ride_length_minutes"] > 0) &
    (rides["ride_length_minutes"] <= 1440)
].copy()

rows_after_duration_cleaning = len(rides)

print("Rows before duration cleaning:", rows_before_duration_cleaning)
print("Rows after duration cleaning:", rows_after_duration_cleaning)
print(
    "Invalid or extreme-duration rows removed:",
    rows_before_duration_cleaning - rows_after_duration_cleaning
)

In [ ]:
print("Rider categories:")
print(rides["member_casual"].value_counts())

print("\nBike types:")
print(rides["rideable_type"].value_counts())

In [ ]:
rider_summary = (
    rides["member_casual"]
    .value_counts()
    .rename_axis("rider_type")
    .reset_index(name="number_of_rides")
)

rider_summary["percentage"] = (
    rider_summary["number_of_rides"] / len(rides) * 100
).round(2)

print(rider_summary)

In [ ]:
ride_length_summary = (
    rides.groupby("member_casual")["ride_length_minutes"]
    .agg(
        number_of_rides="count",
        average_ride_length="mean",
        median_ride_length="median",
        maximum_ride_length="max"
    )
    .round(2)
)

print(ride_length_summary)

In [ ]:
print(ride_length_summary.T)

In [ ]:
day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

rides_by_day = (
    rides.groupby(["day_of_week", "member_casual"])
    .size()
    .reset_index(name="number_of_rides")
)

rides_by_day["day_of_week"] = pd.Categorical(
    rides_by_day["day_of_week"],
    categories=day_order,
    ordered=True
)

rides_by_day = rides_by_day.sort_values(
    ["day_of_week", "member_casual"]
)

print(rides_by_day)

In [ ]:
average_length_by_day = (
    rides.groupby(["day_of_week", "member_casual"])["ride_length_minutes"]
    .mean()
    .reset_index(name="average_ride_length")
)

average_length_by_day["day_of_week"] = pd.Categorical(
    average_length_by_day["day_of_week"],
    categories=day_order,
    ordered=True
)

average_length_by_day = average_length_by_day.sort_values(
    ["day_of_week", "member_casual"]
)

average_length_by_day["average_ride_length"] = (
    average_length_by_day["average_ride_length"].round(2)
)

print(average_length_by_day)

In [ ]:
month_order = [
    "July", "August", "September", "October", "November", "December",
    "January", "February", "March", "April", "May", "June"
]

rides_by_month = (
    rides.groupby(["month", "member_casual"])
    .size()
    .reset_index(name="number_of_rides")
)

rides_by_month["month"] = pd.Categorical(
    rides_by_month["month"],
    categories=month_order,
    ordered=True
)

rides_by_month = rides_by_month.sort_values(
    ["month", "member_casual"]
)

print(rides_by_month)

In [ ]:
rides_by_hour = (
    rides.groupby(["hour", "member_casual"])
    .size()
    .reset_index(name="number_of_rides")
)

print(rides_by_hour)

In [ ]:
peak_hours = (
    rides_by_hour.loc[
        rides_by_hour.groupby("member_casual")["number_of_rides"].idxmax()
    ]
    .sort_values("member_casual")
)

print("Peak hour for each rider group:")
print(peak_hours)

In [ ]:
hourly_pivot = rides_by_hour.pivot(
    index="hour",
    columns="member_casual",
    values="number_of_rides"
)

print(hourly_pivot)

In [ ]:
bike_type_summary = (
    rides.groupby(["rideable_type", "member_casual"])
    .size()
    .reset_index(name="number_of_rides")
)

bike_type_summary["percentage_within_rider_group"] = (
    bike_type_summary.groupby("member_casual")["number_of_rides"]
    .transform(lambda x: x / x.sum() * 100)
    .round(2)
)

print(bike_type_summary)

In [ ]:
cleaned_data_folder = project_folder / "02 Cleaned Data"
cleaned_data_folder.mkdir(parents=True, exist_ok=True)

cleaned_file = cleaned_data_folder / "cyclistic_cleaned_2025_07_to_2026_06.csv"

rides.to_csv(cleaned_file, index=False)

print("Cleaned dataset saved to:")
print(cleaned_file)
print("Final number of rows:", len(rides))
print("Final number of columns:", len(rides.columns))

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
rider_counts = rides["member_casual"].value_counts()

plt.figure(figsize=(7, 5))
plt.bar(rider_counts.index, rider_counts.values)
plt.title("Total Rides by Rider Type")
plt.xlabel("Rider Type")
plt.ylabel("Number of Rides")
plt.tight_layout()
plt.show()

In [ ]:
chart_folder = project_folder / "04 Visualisations"

chart_folder.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(7, 5))
plt.bar(rider_counts.index, rider_counts.values)
plt.title("Total Rides by Rider Type")
plt.xlabel("Rider Type")
plt.ylabel("Number of Rides")
plt.tight_layout()

chart_path = chart_folder / "total_rides_by_rider_type.png"
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()

print("Chart saved to:")
print(chart_path)

In [ ]:
average_lengths = (
    rides.groupby("member_casual")["ride_length_minutes"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(7, 5))
plt.bar(average_lengths.index, average_lengths.values)
plt.title("Average Ride Length by Rider Type")
plt.xlabel("Rider Type")
plt.ylabel("Average Ride Length (Minutes)")
plt.tight_layout()

chart_path = chart_folder / "average_ride_length_by_rider_type.png"
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()

print("Chart saved to:")
print(chart_path)

In [ ]:
day_pivot = rides_by_day.pivot(
    index="day_of_week",
    columns="member_casual",
    values="number_of_rides"
).reindex(day_order)

day_pivot.plot(kind="bar", figsize=(10, 6))

plt.title("Number of Rides by Day of the Week")
plt.xlabel("Day of the Week")
plt.ylabel("Number of Rides")
plt.xticks(rotation=45)
plt.tight_layout()

chart_path = chart_folder / "rides_by_day_of_week.png"
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()

print("Chart saved to:")
print(chart_path)

In [ ]:
month_pivot = rides_by_month.pivot(
    index="month",
    columns="member_casual",
    values="number_of_rides"
).reindex(month_order)

month_pivot.plot(kind="line", marker="o", figsize=(10, 6))

plt.title("Monthly Ridership Trends")
plt.xlabel("Month")
plt.ylabel("Number of Rides")
plt.xticks(rotation=45)
plt.tight_layout()

chart_path = chart_folder / "monthly_ridership_trends.png"
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()

print("Chart saved to:")
print(chart_path)

In [ ]:
hourly_pivot.plot(kind="line", marker="o", figsize=(10, 6))

plt.title("Ridership by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Rides")
plt.xticks(range(0, 24))
plt.tight_layout()

chart_path = chart_folder / "rides_by_hour_of_day.png"
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()

print("Chart saved to:")
print(chart_path)

In [ ]:
bike_type_pivot = bike_type_summary.pivot(
    index="rideable_type",
    columns="member_casual",
    values="percentage_within_rider_group"
)

bike_type_pivot.plot(kind="bar", figsize=(8, 5))

plt.title("Bike Type Preference by Rider Type")
plt.xlabel("Bike Type")
plt.ylabel("Percentage Within Rider Group")
plt.xticks(rotation=0)
plt.tight_layout()

chart_path = chart_folder / "bike_type_preference.png"
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()

print("Chart saved to:")
print(chart_path)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

rider_counts = rides["member_casual"].value_counts()

plt.figure(figsize=(7, 5))
plt.bar(rider_counts.index, rider_counts.values)

plt.title("Total Rides by Rider Type")
plt.xlabel("Rider Type")
plt.ylabel("Number of Rides")

plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{int(x):,}")
)

plt.tight_layout()

chart_path = chart_folder / "total_rides_by_rider_type.png"
plt.savefig(chart_path, dpi=300, bbox_inches="tight")
plt.show()